# XAI Pipeline Latency Analysis — Plots (Interactive View)

Displays the same 7 dissertation figures as `analysis/mann_whitney.py`, for interactive inspection inside the notebook.

**This notebook does not save any files.** `analysis/mann_whitney.py` is the sole source of truth for `results/plots/*.png`, `results/summary_table.csv`, and `results/mann_whitney_report.txt`. Re-run that script (not this notebook) after any change to the underlying CSVs.

**Input:** `results/locust_<arch>_<N>u_run<R>_stats.csv` — up to 3 repeat runs per (architecture × concurrency) combo.

**Three architectures compared:**
- **No XAI** — Pure LightGBM inference, no explanation (control group)
- **Synch XAI** — Synchronous SHAP + counterfactuals in the request cycle
- **Asynch XAI** — Tier-1 SHAP immediate, Tier-2 offloaded to Celery/Redis

**Concurrency levels:** 1, 5, 10, 25, 50 concurrent users (60 seconds each, x3 repeats)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────
ROOT        = Path().resolve().parent
RESULTS_DIR = ROOT / "results"
STATS_DIR   = RESULTS_DIR / "stats"   # only *_stats.csv is read; failures/history/exceptions are kept as raw evidence, unused here
PLOTS_DIR   = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

print(f"Results dir : {RESULTS_DIR}")
print(f"Stats dir   : {STATS_DIR}")
print(f"Plots dir   : {PLOTS_DIR}")

In [2]:
# ── Constants ──────────────────────────────────────────────────────────────
CONCURRENCY = [1, 5, 10, 25, 50]
ARCHS       = ["baseline", "synch", "asynch"]

ENDPOINT_MAP = {
    "baseline": "/predict/baseline",
    "synch":    "/predict/synch",
    "asynch":   "/predict/asynch",
}

LABELS = {
    "baseline": "No XAI",
    "synch":    "Synch XAI",
    "asynch":   "Asynch XAI",
}

COLORS = {
    "baseline": "#22c55e",
    "synch":    "#ef4444",
    "asynch":   "#3b82f6",
}

FILENAME_RE = re.compile(r"locust_(baseline|synch|asynch)_(\d+)u_run\d+_stats\.csv$")

sns.set_theme(style="whitegrid", font_scale=1.1)
print("Constants ready.")

Constants ready.


In [ ]:
# ── Load all CSV data (repeat-aware — mirrors analysis/mann_whitney.py) ─────
def parse_csv(path, arch):
    endpoint = ENDPOINT_MAP[arch]
    df  = pd.read_csv(path)
    row = df[(df["Name"] == endpoint) & (df["Type"].str.upper() == "POST")]
    if row.empty:
        row = df[df["Name"] == "Aggregated"]
    if row.empty:
        return None
    r     = row.iloc[0]
    total = int(r["Request Count"])
    fails = int(r["Failure Count"])
    return {
        "p50":          float(r["50%"]),
        "p95":          float(r["95%"]),
        "p99":          float(r["99%"]),
        "req_s":        float(r["Requests/s"]),
        "failure_rate": round(fails / total * 100, 1) if total > 0 else 0.0,
        "request_count": total,
        "failure_count": fails,
    }

def average_runs(runs):
    """Aggregate repeat runs at one concurrency level into a single summary point."""
    total = sum(r["request_count"] for r in runs)
    fails = sum(r["failure_count"] for r in runs)
    return {
        "p50":           float(np.mean([r["p50"] for r in runs])),
        "p95":           float(np.mean([r["p95"] for r in runs])),
        "p95_std":       float(np.std([r["p95"] for r in runs])),
        "p99":           float(np.mean([r["p99"] for r in runs])),
        "req_s":         float(np.mean([r["req_s"] for r in runs])),
        "failure_rate":  round(fails / total * 100, 1) if total > 0 else 0.0,
        "request_count": total,
        "failure_count": fails,
        "n_runs":        len(runs),
    }

# raw_data: every individual repeat run — {arch: {n: [run1_dict, run2_dict, run3_dict]}}
raw_data = {arch: {} for arch in ARCHS}
for arch in ARCHS:
    for path in sorted(STATS_DIR.glob(f"locust_{arch}_*_stats.csv")):
        m = FILENAME_RE.search(path.name)
        if m:
            n = int(m.group(2))
            d = parse_csv(path, arch)
            if d:
                raw_data[arch].setdefault(n, []).append(d)

# all_data: one averaged point per (arch, concurrency) — used for the trend plots (fig1-4, 6, 7)
all_data = {
    arch: {n: average_runs(runs) for n, runs in raw_data[arch].items()}
    for arch in ARCHS
}

# p95_samples: every individual repeat run's p95 value, flattened across concurrency levels —
# used for the box plot (fig5), matching what actually feeds the Mann-Whitney tests
p95_samples = {
    arch: [run["p95"] for n in CONCURRENCY if n in raw_data[arch] for run in raw_data[arch][n]]
    for arch in ARCHS
}

# Summary table
rows = []
for arch in ARCHS:
    for n in CONCURRENCY:
        d = all_data[arch].get(n)
        if d:
            rows.append({"Architecture": LABELS[arch], "Users": n,
                         "p50 (ms)": int(d["p50"]), "p95 (ms)": int(d["p95"]),
                         "p99 (ms)": int(d["p99"]), "Req/s": round(d["req_s"],2),
                         "Fail%": d["failure_rate"], "Repeats": d["n_runs"]})

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

## Figure 1 — p95 Tail Latency vs Concurrent Users
The primary dissertation figure. Log scale is necessary because the range spans 130 ms (baseline, 1 user) to 36,000 ms (synch, 50 users). On a linear scale, the baseline would be invisible at the bottom.

**Reading the chart:** Asynch XAI tracks close to No XAI at low concurrency (1–5 users), then diverges at 25–50 users as Tier-1 SHAP itself becomes a bottleneck under sustained load. Synch XAI degrades catastrophically from 1 user onwards.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for arch in ARCHS:
    xs = [n for n in CONCURRENCY if n in all_data[arch]]
    ys = [all_data[arch][n]["p95"] for n in xs]
    errs = [all_data[arch][n].get("p95_std", 0) for n in xs]
    ax.errorbar(xs, ys, yerr=errs, marker="o", linewidth=2.5, markersize=8,
                capsize=4, color=COLORS[arch], label=LABELS[arch])
    for x, y in zip(xs, ys):
        ax.annotate(f"{int(y):,} ms",           # ← int() removes trailing .0
                    xy=(x, y), xytext=(5, 7),
                    textcoords="offset points",
                    fontsize=8, color=COLORS[arch])

# Shade the "acceptable" zone (< 1000ms) to give visual reference
ax.axhspan(0, 1000, alpha=0.04, color="green", label="< 1,000 ms zone")
ax.axhline(1000, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax.text(50.5, 1050, "1,000 ms", fontsize=8, color="gray", va="bottom")

ax.set_yscale("log")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v):,} ms"))
ax.set_xticks(CONCURRENCY)
ax.set_xlabel("Concurrent Users", fontsize=12)
ax.set_ylabel("p95 Tail Latency (log scale)", fontsize=12)
ax.set_title("p95 Tail Latency vs Concurrent Users  (error bars = std across repeat runs)", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## Figure 2 — p50 / p95 / p99 Percentile Comparison
Shows the full tail behaviour across all three latency percentiles. The growing gap between p95 and p99 for Synch XAI at high concurrency indicates heavy tail inflation — a small fraction of users wait significantly longer than the median.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharey=False)
metrics = [("p50", "p50  (Median)"), ("p95", "p95  (Tail)"), ("p99", "p99  (Extreme Tail)")]

for ax, (metric, title) in zip(axes, metrics):
    for arch in ARCHS:
        xs = [n for n in CONCURRENCY if n in all_data[arch]]
        ys = [all_data[arch][n][metric] for n in xs]
        ax.plot(xs, ys, marker="o", linewidth=2, markersize=6,
                color=COLORS[arch], label=LABELS[arch])
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax.set_xticks(CONCURRENCY)
    ax.set_xlabel("Concurrent Users", fontsize=11)
    ax.set_ylabel("Latency (ms)", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)

fig.suptitle("Latency Percentile Comparison — All Architectures",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Figure 3 — System Throughput vs Concurrent Users
Throughput (requests/second) shows how many users each architecture can actually serve. Synch XAI collapses to under 2 req/s at 50 users because each request occupies a worker for 17+ seconds. No XAI scales linearly because inference alone is fast.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for arch in ARCHS:
    xs = [n for n in CONCURRENCY if n in all_data[arch]]
    ys = [all_data[arch][n]["req_s"] for n in xs]
    ax.plot(xs, ys, marker="s", linewidth=2.2, markersize=7,
            color=COLORS[arch], label=LABELS[arch])
    for x, y in zip(xs, ys):
        ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(4, 6),
                    textcoords="offset points", fontsize=8, color=COLORS[arch])

ax.set_xticks(CONCURRENCY)
ax.set_xlabel("Concurrent Users", fontsize=12)
ax.set_ylabel("Throughput (Requests / Second)", fontsize=12)
ax.set_title("System Throughput vs Concurrent Users", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Figure 4 — Request Failure Rate vs Concurrent Users
Synch XAI failure rate reaches **78.8% at 50 users** — the system is effectively broken under realistic concurrent load. Asynch XAI degrades more gracefully. No XAI remains near-zero throughout.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for arch in ARCHS:
    xs = [n for n in CONCURRENCY if n in all_data[arch]]
    ys = [all_data[arch][n]["failure_rate"] for n in xs]
    ax.plot(xs, ys, marker="^", linewidth=2.2, markersize=7,
            color=COLORS[arch], label=LABELS[arch])
    for x, y in zip(xs, ys):
        if y > 0:
            ax.annotate(f"{y:.1f}%", xy=(x, y), xytext=(4, 6),
                        textcoords="offset points", fontsize=9, color=COLORS[arch])

ax.set_xticks(CONCURRENCY)
ax.set_ylim(-3, 92)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.set_xlabel("Concurrent Users", fontsize=12)
ax.set_ylabel("Request Failure Rate", fontsize=12)
ax.set_title("Request Failure Rate vs Concurrent Users", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Figure 5 — p95 Distribution Box Plots (Mann-Whitney U Input)
Each box represents the distribution of p95 values across the five concurrency levels (1, 5, 10, 25, 50 users). Individual dots show all five data points. This directly visualises the input to the Mann-Whitney U tests — the separation between boxes demonstrates why the tests reach significance.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
data   = [p95_samples[a] for a in ARCHS]
labels = [LABELS[a] for a in ARCHS]
colors = [COLORS[a] for a in ARCHS]

bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                medianprops=dict(color="black", linewidth=2.5))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

rng = np.random.default_rng(42)
for i, (vals, color) in enumerate(zip(data, colors), start=1):
    jitter = rng.uniform(-0.07, 0.07, len(vals))
    ax.scatter([i + j for j in jitter], vals,
               color=color, zorder=5, s=60, edgecolors="white", linewidths=0.8)

ax.set_yscale("log")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v):,} ms"))
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("p95 Tail Latency (log scale)", fontsize=12)
ax.set_title(
    f"p95 Latency Distribution Across Concurrency Levels\n"
    f"(Each dot = one repeat run, {len(data[0])} total per architecture)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

## Figure 6 — p95 Latency Heatmap (Architecture × Concurrency)
A colour-coded matrix giving an instant visual overview. Dark green = fast (safe), dark red = slow (SLA violation). Values shown in seconds for readability.

In [ ]:
# Build matrix (values in SECONDS for short, readable annotations)
matrix = []
for arch in ARCHS:
    row = [all_data[arch].get(n, {}).get("p95", float("nan")) / 1000
           for n in CONCURRENCY]
    matrix.append(row)

df_heat = pd.DataFrame(
    matrix,
    index=[LABELS[a] for a in ARCHS],
    columns=[f"{n} users" for n in CONCURRENCY]
)

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(
    df_heat,
    annot=True,
    fmt=".2f",                          # e.g. "36.00" instead of "36000"
    cmap="RdYlGn_r",
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 11, "weight": "bold"},
    cbar_kws={"label": "p95 Latency (seconds)", "shrink": 0.8}
)

# Fix Y-axis label overlap: rotate to horizontal and add right-padding
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, va="center", fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=10)
ax.set_ylabel("")          # label is self-evident from row names
ax.set_xlabel("Concurrent Users", fontsize=11, labelpad=8)
ax.set_title(
    "p95 Tail Latency Heatmap  (values in seconds)\n"
    "Architecture × Concurrent Users",
    fontsize=13, fontweight="bold", pad=12
)

plt.tight_layout()
plt.show()

## Figure 7 — Asynch XAI Speedup Over Synch XAI
Grouped bar chart on a linear scale (bars start accurately from zero) showing actual p95 latency for Synch XAI vs Asynch XAI. The ×N label above each Asynch bar shows how many times faster the async pipeline is. The visual height difference between red and blue bars directly communicates the scale of improvement.

In [ ]:
xs, ratios, synch_s_vals, asynch_s_vals = [], [], [], []
for n in CONCURRENCY:
    s = all_data["synch"].get(n, {}).get("p95")
    a = all_data["asynch"].get(n, {}).get("p95")
    if s and a and a > 0:
        xs.append(n)
        ratios.append(round(s / a, 1))
        synch_s_vals.append(s / 1000)
        asynch_s_vals.append(a / 1000)

fig, ax = plt.subplots(figsize=(10, 6))

x      = np.arange(len(xs))
width  = 0.35
y_max  = max(synch_s_vals) * 1.18   # headroom for labels

bars_s = ax.bar(x - width / 2, synch_s_vals, width,
                color=COLORS["synch"], alpha=0.85, label="Synch XAI",
                edgecolor="white", linewidth=0.5)
bars_a = ax.bar(x + width / 2, asynch_s_vals, width,
                color=COLORS["asynch"], alpha=0.85, label="Asynch XAI",
                edgecolor="white", linewidth=0.5)

# Actual latency label above each bar
for bar, val in zip(bars_s, synch_s_vals):
    lbl = f"{val:.1f}s" if val >= 1 else f"{val:.2f}s"
    ax.text(bar.get_x() + bar.get_width() / 2, val + y_max * 0.01,
            lbl, ha="center", va="bottom", fontsize=8.5,
            color=COLORS["synch"], fontweight="bold")

for bar, val in zip(bars_a, asynch_s_vals):
    lbl = f"{val:.2f}s" if val < 1 else f"{val:.1f}s"
    ax.text(bar.get_x() + bar.get_width() / 2, val + y_max * 0.01,
            lbl, ha="center", va="bottom", fontsize=8.5,
            color=COLORS["asynch"], fontweight="bold")

# ×N speedup label above each Asynch bar
for i, (ratio, a_val) in enumerate(zip(ratios, asynch_s_vals)):
    ax.text(x[i] + width / 2, a_val + y_max * 0.09,
            f"{ratio:.1f}×", ha="center", va="bottom",
            fontsize=11, fontweight="bold", color="#333",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                      alpha=0.92, edgecolor="#ccc", linewidth=0.6))

ax.set_xticks(x)
ax.set_xticklabels([str(n) for n in xs])
ax.set_ylim(0, y_max)
ax.set_xlabel("Concurrent Users", fontsize=12)
ax.set_ylabel("p95 Latency (seconds)", fontsize=12)
ax.set_title(
    "Benefit of Asynchronous Decoupling\n"
    "p95 Latency: Synch XAI vs Asynch XAI  (×N = speedup factor)",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

## Summary

These are the same 7 figures `analysis/mann_whitney.py` saves to `results/plots/` — this notebook only displays them interactively and writes no files itself.

| Figure | File (saved by `mann_whitney.py`) | Dissertation use |
|--------|------|------------------|
| Fig 1 | `fig1_p95_latency.png` | Primary results figure |
| Fig 2 | `fig2_percentile_comparison.png` | Full tail behaviour |
| Fig 3 | `fig3_throughput.png` | Capacity / SLA argument |
| Fig 4 | `fig4_failure_rate.png` | Reliability degradation |
| Fig 5 | `fig5_boxplots.png` | Mann-Whitney U visualisation |
| Fig 6 | `fig6_p95_heatmap.png` | Quick overview / presentation |
| Fig 7 | `fig7_async_speedup.png` | Quantifies the async advantage |

In [ ]:
import os
plots = sorted(PLOTS_DIR.glob("*.png"))
print(f"{len(plots)} plots currently in {PLOTS_DIR} (last saved by analysis/mann_whitney.py):")
for p in plots:
    print(f"  {p.name}  ({os.path.getsize(p):,} bytes)")